In [17]:
import hopsworks
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

# ── Connect ────────────────────────────────────────────────────────────────────
project = hopsworks.login(
    host=os.getenv("HOPSWORKS_HOST", "eu-west.cloud.hopsworks.ai"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)
fs      = project.get_feature_store()
fg      = fs.get_feature_group(name="aqi_features", version=1)
df      = fg.read(online=False)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df      = df.sort_values("timestamp").reset_index(drop=True)

print(f"Total rows:  {len(df)}")
print(f"Date range:  {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Columns:     {len(df.columns)}")

# ── Basic stats ────────────────────────────────────────────────────────────────
print("\n--- AQI stats ---")
print(df["aqi"].describe().round(2))

# ── Check forecast column coverage ────────────────────────────────────────────
fc_cols = [c for c in df.columns if c.startswith("fc_")]
print(f"\n--- Forecast columns ({len(fc_cols)}) ---")
print(df[fc_cols].notna().sum().to_string())

# ── Rows WITH forecast features ───────────────────────────────────────────────
rows_with_fc = df[df["fc_pm25_24h"].notna()]
print(f"\n--- Rows with forecast features: {len(rows_with_fc)} ---")
print(rows_with_fc[["timestamp", "aqi", "fc_pm25_24h", "fc_dust_48h", "fc_uvi_72h"]].tail(10).to_string())

# ── Rows with real AQI (training eligible) ────────────────────────────────────
real_aqi = df[df["aqi"].notna()]
print(f"\n--- Rows with real AQI (training eligible): {len(real_aqi)} ---")
print(real_aqi[["timestamp", "aqi", "pm25", "pm10", "rolling_mean_24h"]].tail(10).to_string())

# ── Check NaN counts per column ───────────────────────────────────────────────
print("\n--- NaN counts per column ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string())

# ── AQI distribution by year ───────────────────────────────────────────────────
print("\n--- AQI mean by year ---")
print(df.groupby(df["timestamp"].dt.year)["aqi"].agg(["mean", "count", "min", "max"]).round(2))

# ── Latest 5 rows ─────────────────────────────────────────────────────────────
print("\n--- Latest 5 rows ---")
print(df.tail(5)[["timestamp", "aqi", "pm25", "pm10", "aqi_lag_1h", 
                   "rolling_mean_24h", "fc_pm25_24h", "fc_dust_48h"]].to_string())

# ── Earliest 5 rows ───────────────────────────────────────────────────────────
print("\n--- Earliest 5 rows ---")
print(df.head(5)[["timestamp", "aqi", "pm25", "pm10", "aqi_lag_1h",
                   "rolling_mean_24h", "fc_pm25_24h"]].to_string())

2026-04-28 15:56:30,358 INFO: Closing external client and cleaning up certificates.
2026-04-28 15:56:30,361 INFO: Connection closed.
2026-04-28 15:56:30,362 INFO: Initializing external client
2026-04-28 15:56:30,363 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-04-28 15:56:32,499 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31957
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.74s) 
Total rows:  2232
Date range:  2026-01-26 00:00:00+00:00 → 2026-04-28 23:00:00+00:00
Columns:     48

--- AQI stats ---
count    2232.00
mean       27.47
std        11.24
min         3.50
25%        19.80
50%        25.40
75%        32.90
max        79.90
Name: aqi, dtype: float64

--- Forecast columns (21) ---
fc_pm25_24h    2232
fc_co_24h      2232
fc_no2_24h     2232
fc_so2_24h     2232
fc_o3_24h      2232
fc_dust_24h      47
fc_uvi_24h       47
fc_pm25_48h    2232
fc_co_48h      2232
fc_no2_